# Project 2: House Price Prediction

**InternCareerPath – Machine Learning (AI) Self-Learning Internship**

This project predicts California house values using regression models. It compares Linear Regression and Random Forest Regression, performs feature engineering, cross-validation, hyperparameter tuning, and error analysis.


## Objectives
- Prepare and inspect a regression dataset.
- Engineer useful features.
- Train Linear Regression and Random Forest models.
- Evaluate models using MAE and RMSE.
- Use cross-validation and hyperparameter tuning.
- Analyze prediction errors.
- Save the best model for later deployment in Project 8.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


## 1. Load the Dataset

The California Housing dataset contains numerical housing and demographic features. The target is median house value.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

df.head()

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:\n', df.isnull().sum())
df.describe()

## 2. Feature Engineering

Ratios can provide more useful information than raw counts. We create rooms-per-household and bedrooms-per-room while protecting against division by zero.

In [ ]:
df['RoomsPerHousehold'] = df['AveRooms'] / df['AveOccup'].replace(0, np.nan)
df['BedroomsPerRoom'] = df['AveBedrms'] / df['AveRooms'].replace(0, np.nan)
df = df.replace([np.inf, -np.inf], np.nan).dropna()

X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

print('Features:', list(X.columns))
print('Rows after preprocessing:', len(df))

## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## 4. Linear Regression Baseline

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_r2 = r2_score(y_test, linear_pred)

print(f'Linear Regression MAE:  {linear_mae:.4f}')
print(f'Linear Regression RMSE: {linear_rmse:.4f}')
print(f'Linear Regression R²:   {linear_r2:.4f}')

## 5. Random Forest Regression

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print(f'Random Forest MAE:  {rf_mae:.4f}')
print(f'Random Forest RMSE: {rf_rmse:.4f}')
print(f'Random Forest R²:   {rf_r2:.4f}')

## 6. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE': [linear_mae, rf_mae],
    'RMSE': [linear_rmse, rf_rmse],
    'R2': [linear_r2, rf_r2]
})
results.sort_values('RMSE')

## 7. Cross-Validation

Five-fold cross-validation is used to check whether the Random Forest performance is consistent across different training/validation splits.

In [ ]:
cv_scores = cross_val_score(
    rf_model, X_train, y_train,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

print('CV RMSE scores:', -cv_scores)
print(f'Mean CV RMSE: {-cv_scores.mean():.4f}')

## 8. Hyperparameter Tuning

GridSearchCV tests a small set of Random Forest configurations and selects the one with the lowest cross-validated RMSE.

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 15, 25],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print('Best parameters:', grid_search.best_params_)
print('Best CV RMSE:', -grid_search.best_score_)

## 9. Final Evaluation

In [ ]:
best_pred = best_model.predict(X_test)

final_mae = mean_absolute_error(y_test, best_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, best_pred))
final_r2 = r2_score(y_test, best_pred)

print(f'Final MAE:  {final_mae:.4f}')
print(f'Final RMSE: {final_rmse:.4f}')
print(f'Final R²:   {final_r2:.4f}')

## 10. Error Analysis

Residuals are the difference between actual and predicted values. We inspect their distribution and the relationship between actual and predicted prices.

In [ ]:
residuals = y_test - best_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_test, best_pred, alpha=0.35)
plt.xlabel('Actual House Value')
plt.ylabel('Predicted House Value')
plt.title('Actual vs Predicted House Values')
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=40)
plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Frequency')
plt.title('Residual Distribution')
plt.show()

print('Mean residual:', residuals.mean())
print('Largest absolute errors:')
print(residuals.abs().sort_values(ascending=False).head())

## 11. Feature Importance

Random Forest provides an estimate of the relative importance of each feature.

In [ ]:
importance = pd.Series(
    best_model.feature_importances_, index=X.columns
).sort_values(ascending=False)

print(importance)

plt.figure(figsize=(9, 5))
importance.sort_values().plot(kind='barh')
plt.xlabel('Importance')
plt.title('Random Forest Feature Importance')
plt.show()

## 12. Save the Model

The saved model will be reused in Project 8, where it will be exposed through a FastAPI endpoint.

In [ ]:
joblib.dump(best_model, 'house_price_model.joblib')
print('Saved: house_price_model.joblib')

## Conclusion

The project compared a simple Linear Regression baseline with Random Forest Regression. Feature engineering, cross-validation, and hyperparameter tuning were applied to improve reliability. The final model was evaluated using MAE, RMSE, and R², followed by residual and feature-importance analysis. The trained model is saved for use in Project 8.